In [4]:
import os

# Set working directory to project root always
# Works regardless of where the notebook is saved
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
os.chdir(project_root)

print("Working directory set to:", os.getcwd())

Working directory set to: c:\Users\DELL\OneDrive\Documents\SRM\Churn_Analysis


In [5]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv(
    'data/healthcare/health_churn.csv'
)

print(df.shape)
df.head()

(2000, 21)


,PatientID,Age,Gender,State,Tenure_Months,Specialty,Insurance_Type,Visits_Last_Year,Missed_Appointments,Days_Since_Last_Visit,...,Overall_Satisfaction,Wait_Time_Satisfaction,Staff_Satisfaction,Provider_Rating,Avg_Out_Of_Pocket_Cost,Billing_Issues,Portal_Usage,Referrals_Made,Distance_To_Facility_Miles,Churned
0,C20000,41,Female,PA,62,Pediatrics,Medicaid,1,0,564,...,3.5,4.9,3.8,4.2,306,0,0,3,21.4,1
1,C20001,43,Female,GA,44,Internal Medicine,Self-Pay,7,4,254,...,2.6,3.1,4.7,4.3,1851,0,0,0,47.6,1
2,C20002,21,Male,MI,120,Internal Medicine,Medicaid,15,5,89,...,1.6,4.4,2.1,4.7,391,0,0,2,7.1,0
3,C20003,65,Male,FL,118,General Practice,Private,10,3,135,...,2.6,4.3,4.3,4.9,808,0,0,0,11.6,1
4,C20004,18,Female,CA,70,Cardiology,Medicaid,5,4,696,...,2.2,4.0,4.1,4.4,866,0,0,0,10.3,1


In [6]:
df.drop(
    columns=[
        'PatientID',
        'Last_Interaction_Date'
    ],
    inplace=True
)

print(df.shape)

(2000, 19)


In [7]:
print(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

Age                           0
Gender                        0
State                         0
Tenure_Months                 0
Specialty                     0
Insurance_Type                0
Visits_Last_Year              0
Missed_Appointments           0
Days_Since_Last_Visit         0
Overall_Satisfaction          0
Wait_Time_Satisfaction        0
Staff_Satisfaction            0
Provider_Rating               0
Avg_Out_Of_Pocket_Cost        0
Billing_Issues                0
Portal_Usage                  0
Referrals_Made                0
Distance_To_Facility_Miles    0
Churned                       0
dtype: int64


In [8]:
for col in df.select_dtypes(
    include=['int64','float64']
).columns:

    df[col].fillna(
        df[col].median(),
        inplace=True
    )

for col in df.select_dtypes(
    include='object'
).columns:

    df[col].fillna(
        df[col].mode()[0],
        inplace=True
    )

print(
    "Remaining Nulls:",
    df.isnull().sum().sum()
)

Remaining Nulls: 0


C:\Users\DELL\AppData\Local\Temp\ipykernel_19972\790537524.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(
C:\Users\DELL\AppData\Local\Temp\ipykernel_19972\790537524.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplac

In [9]:
print(
    df['Churned']
    .value_counts()
)

Churned
1    1367
0     633
Name: count, dtype: int64


In [10]:
cat_cols = (
    df.select_dtypes(
        include='object'
    )
    .columns
    .tolist()
)

print(cat_cols)

['Gender', 'State', 'Specialty', 'Insurance_Type']


C:\Users\DELL\AppData\Local\Temp\ipykernel_19972\2458213787.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(


In [11]:
le = LabelEncoder()

for col in cat_cols:

    df[col] = (
        le.fit_transform(
            df[col].astype(str)
        )
    )

    print(
        f"Encoded: {col}"
    )

Encoded: Gender
Encoded: State
Encoded: Specialty
Encoded: Insurance_Type


In [12]:
print(
    "\nObject Columns Left:"
)

print(
    df.select_dtypes(
        include='object'
    ).columns.tolist()
)

print(
    "\nFinal Shape:",
    df.shape
)


Object Columns Left:
[]

Final Shape: (2000, 19)


In [13]:
import os

os.makedirs(
    'outputs',
    exist_ok=True
)

df.to_csv(
    'outputs/healthcare_clean.csv',
    index=False
)

print(
    "Saved successfully"
)

Saved successfully


In [14]:
import os
import joblib
from sklearn.preprocessing import LabelEncoder

cat_cols = (
    df.select_dtypes(
        include='object'
    )
    .columns
    .tolist()
)

encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    encoders[col] = le

    print(
        f"Encoded: {col}"
    )

os.makedirs(
    'outputs',
    exist_ok=True
)

joblib.dump(
    encoders,
    'outputs/healthcare_encoders.pkl'
)

print(
    "Healthcare encoders saved successfully"
)

Healthcare encoders saved successfully
